<a href="https://www.kaggle.com/code/hossammostafa25/project?scriptVersionId=315406763" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# YOLOv8 Vehicle Detection - Full Dataset Pipeline (Kaggle)
This notebook implements a complete YOLOv8 pipeline optimized for Kaggle. It automatically handles dataset extraction, detects structure, and trains on the **full dataset** without any sampling.

In [ ]:
!pip install ultralytics opencv-python --quiet


In [ ]:
import os
import random
import shutil
import zipfile
import yaml
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Automatically detect GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Execution Device: {'GPU' if device == 0 else 'CPU'}")

## 1. Dataset Handling
Extracting and preparing the full dataset from `/kaggle/input/my-dataset/dataset.zip`.

In [ ]:
# Path to your dataset based on the screenshot
dataset_input_dir = Path('/kaggle/input/datasets/hossammostafa25/hosssssss')

if not dataset_input_dir.exists():
    # Fallback search if the name differs slightly
    print("Target path not found, searching /kaggle/input...")
    potential_paths = list(Path('/kaggle/input').rglob('train'))
    if potential_paths:
        dataset_input_dir = potential_paths[0].parent
    else:
        raise FileNotFoundError("Could not find dataset in /kaggle/input. Please check the 'Input' section on the right.")

print(f"Dataset found at: {dataset_input_dir}")

# Load class names from existing data.yaml
class_names = {0: 'vehicle'} # Default
existing_yaml = dataset_input_dir / 'data.yaml'
if existing_yaml.exists():
    with open(existing_yaml, 'r') as f:
        data = yaml.safe_load(f)
        if 'names' in data: 
            class_names = data['names']
            print("Loaded class names from dataset data.yaml.")

# Create the Kaggle-specific data.yaml in the working directory
# We map 'valid' to 'val' as YOLOv8 expects
data_yaml_content = {
    'path': str(dataset_input_dir.resolve()),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': class_names
}

working_yaml = '/kaggle/working/data.yaml'
with open(working_yaml, 'w') as f:
    yaml.dump(data_yaml_content, f)

print(f"Configuration saved to {working_yaml}")

## 2. Model Training
Training YOLOv8n on the full dataset for 30 and 50 epochs.

In [ ]:
PARAMS = {'data': working_yaml, 'imgsz': 640, 'batch': 16, 'device': device, 'project': '/kaggle/working/runs', 'seed': SEED}

print("\n--- Training: 30 Epochs ---")
model_30 = YOLO('yolov8n.pt')
model_30.train(epochs=30, name='exp_30', **PARAMS)

print("\n--- Training: 50 Epochs ---")
model_50 = YOLO('yolov8n.pt')
model_50.train(epochs=50, name='exp_50', **PARAMS)

## 3. Evaluation
Extracting and comparing metrics.

In [ ]:
def get_metrics(path):
    model = YOLO(path)
    m = model.val(data=working_yaml, device=device)
    return m.results_dict

m30 = get_metrics('/kaggle/working/runs/exp_30/weights/best.pt')
m50 = get_metrics('/kaggle/working/runs/exp_50/weights/best.pt')

labels = ['Precision', 'Recall', 'mAP@50', 'mAP@50-95']
v30 = [m30['metrics/precision(B)'], m30['metrics/recall(B)'], m30['metrics/mAP50(B)'], m30['metrics/mAP50-95(B)']]
v50 = [m50['metrics/precision(B)'], m50['metrics/recall(B)'], m50['metrics/mAP50(B)'], m50['metrics/mAP50-95(B)']]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, v30, width, label='30 Epochs')
ax.bar(x + width/2, v50, width, label='50 Epochs')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Performance Comparison (Full Dataset)')
ax.legend()
plt.show()

## 4. Inference and Traffic Analysis

In [ ]:
class TrafficDetector:
    def __init__(self, model_path):
        self.model = YOLO(model_path)

    def process(self, img_path):
        res = self.model.predict(img_path, conf=0.25, verbose=False)[0]
        count = len(res.boxes)
        density = "LOW" if count <= 5 else "MEDIUM" if count <= 15 else "HIGH"
        return count, density, res.plot()

detector = TrafficDetector('/kaggle/working/runs/exp_50/weights/best.pt')
test_dir = dataset_input_dir / 'test/images'
test_imgs = list(test_dir.glob('*.jpg')) + list(test_dir.glob('*.png'))

if test_imgs:
    for i in range(min(5, len(test_imgs))):
        count, density, annotated = detector.process(str(test_imgs[i]))
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title(f"Count: {count} | Density: {density}")
        plt.axis('off')
        plt.show()
else:
    print("No test images found.")

In [ ]:
# ==============================
# Export All Results
# ==============================
import json
from pathlib import Path
import shutil
import cv2

# Create output folder
output_dir = Path('/kaggle/working/final_output')
output_dir.mkdir(exist_ok=True)

print("Saving results to:", output_dir)

# ------------------------------
# 1. Save Metrics
# ------------------------------
results = {
    "30_epochs": m30,
    "50_epochs": m50
}

with open(output_dir / 'metrics.json', 'w') as f:
    json.dump(results, f, indent=4)

print("Metrics saved")

# ------------------------------
# 2. Save Comparison Graph
# ------------------------------
fig.savefig(output_dir / 'comparison.png')
print("Graph saved")

# ------------------------------
# 3. Save Inference Images
# ------------------------------
for i in range(min(5, len(test_imgs))):
    count, density, annotated = detector.process(str(test_imgs[i]))

    save_path = output_dir / f"result_{i}.jpg"
    cv2.imwrite(str(save_path), annotated)

print("Inference images saved")

# ------------------------------
# 4. Copy Training Runs (models + logs)
# ------------------------------
runs_src = Path('/kaggle/working/runs')

if runs_src.exists():
    shutil.copytree(runs_src, output_dir / 'runs', dirs_exist_ok=True)
    print("Training runs copied")

# ------------------------------
# 5. Copy data.yaml
# ------------------------------
shutil.copy(working_yaml, output_dir / 'data.yaml')

# ------------------------------
# 6. Create ZIP file
# ------------------------------
zip_path = '/kaggle/working/project_results.zip'
shutil.make_archive('/kaggle/working/project_results', 'zip', output_dir)

print("ZIP file created at:", zip_path)

In [ ]:
from IPython.display import FileLink

FileLink('/kaggle/working/project_results.zip')